# Woche 5 · Transformer und Attention: Wie kommt Kontext in die Vektoren?
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

Letzte Woche hatte „Lager“ einen einzigen Vektor, egal ob Kugellager oder Hochregallager. Heute baut ihr den Mechanismus, der das ändert: **Attention**. Danach schaut ihr in ein echtes Sprachmodell (GPT-2) hinein und lasst es Text erzeugen.

**Lernziele**
- Scaled Dot-Product Attention erklären und mit NumPy implementieren
- Q, K und V als gelernte Projektionen deuten, die kausale Maske verstehen
- Begründen, warum durch $\sqrt{d_k}$ geteilt wird
- Softmax mit Temperatur, Entropie und autoregressive Generierung anwenden

**Aufbau:** Teil A Attention mit NumPy · Teil B GPT-2 von innen
Pflicht: TODO 1–6, Ziel: TODO 7–8. Ein API-Key wird nur für die optionale Vergleichszelle am Ende gebraucht.

In [ ]:
# Setup: GPT-2 wird beim ersten Laden heruntergeladen (ca. 0,5 GB). Transformers und PyTorch sind in Colab vorinstalliert.
%pip install -q openai==3.24.0

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def check(name, test):
    """Selbsttest: zeigt ✅ oder ❌. Fehler im Test gelten als 'noch nicht gelöst'."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

---
# Teil A · Attention mit NumPy

## A1 · Softmax für Matrizen

In Woche 3 habt ihr die Softmax für einen Vektor geschrieben. Attention braucht sie **zeilenweise** für eine ganze Matrix: Jede Zeile wird für sich zu einer Wahrscheinlichkeitsverteilung.

### TODO 1 · Zeilenweise Softmax
Tipp: `axis=-1` und `keepdims=True` bei `max` und `sum`. Dann funktioniert die Funktion für Vektoren und Matrizen.

In [ ]:
def softmax(Z):
    # TODO 1: numerisch stabil (Maximum je Zeile abziehen), dann je Zeile normieren
    pass

In [ ]:
Z = np.array([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0], [1000.0, 1000.0, 999.0]])
check("TODO 1a jede Zeile summiert zu 1", lambda: np.allclose(softmax(Z).sum(axis=1), 1))
check("TODO 1b gleiche Werte → gleichverteilt", lambda: np.allclose(softmax(Z)[1], 1 / 3))
check("TODO 1c stabil bei großen Zahlen", lambda: np.all(np.isfinite(softmax(Z))))
check("TODO 1d klappt auch für Vektoren", lambda: math.isclose(softmax(np.array([2.0, 1.0, 0.1]))[0], 0.659, rel_tol=1e-3))

## A2 · Scaled Dot-Product Attention

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

In Worten: Jedes Token vergleicht seine **Query** mit den **Keys** aller Tokens (Skalarprodukt), skaliert, macht daraus per Softmax Gewichte und bildet die gewichtete Summe der **Values**.

### TODO 2 · Attention-Funktion
`attention(Q, K, V, maske=None)` gibt `(ausgabe, gewichte)` zurück. Die Maske ignorieren wir erst in A3, lasst den `if`-Block zunächst stehen.

In [ ]:
def attention(Q, K, V, maske=None):
    d_k = K.shape[-1]
    # TODO 2a: scores = Q K^T / sqrt(d_k)
    scores = None
    if maske is not None:
        scores = np.where(maske, scores, -np.inf)    # verbotene Positionen bekommen -∞ (A3)
    # TODO 2b: gewichte per softmax, ausgabe = gewichte @ V
    gewichte = None
    ausgabe = None
    return ausgabe, gewichte

**Die Papierübung aus dem Input.** Zwei Dimensionen: Achse 1 steht für „Maschine“, Achse 2 für „Logistik“. „Lager“ ist mehrdeutig und hat den Vektor $(1, 1)$.
Zur Vereinfachung gilt $Q = K = V = X$ (keine Projektionen).

In [ ]:
WOERTER_1 = ["Spindel", "Lager", "heiß"];  X1 = np.array([[2.0, 0.0], [1.0, 1.0], [1.0, 0.0]])
WOERTER_2 = ["Regal",   "Lager", "voll"];  X2 = np.array([[0.0, 2.0], [1.0, 1.0], [0.0, 1.0]])

aus_1, gew_1 = attention(X1, X1, X1)
aus_2, gew_2 = attention(X2, X2, X2)
if gew_1 is not None:
    print("Gewichte von „Lager“ in Satz 1:", dict(zip(WOERTER_1, np.round(gew_1[1], 3))))
    print("Neuer Vektor für „Lager“ in Satz 1:", np.round(aus_1[1], 3), " (vorher [1, 1])")
    print("Neuer Vektor für „Lager“ in Satz 2:", np.round(aus_2[1], 3))

In [ ]:
check("TODO 2a Gewichte stimmen mit der Handrechnung", lambda: np.allclose(gew_1[1], [0.401, 0.401, 0.198], atol=1e-3))
check("TODO 2b Satz 1: Lager rückt Richtung Maschine", lambda: np.allclose(aus_1[1], [1.401, 0.401], atol=1e-3))
check("TODO 2c Satz 2: Lager rückt Richtung Logistik", lambda: np.allclose(aus_2[1], [0.401, 1.401], atol=1e-3))

> **Das ist der Kern:** Dasselbe Wort mit demselben Startvektor bekommt je nach Kontext einen anderen Ausgabevektor. Genau das fehlte den statischen Embeddings aus Woche 4.

## A3 · Die kausale Maske

Ein Sprachmodell wie GPT-2 soll das **nächste** Token vorhersagen. Beim Training darf ein Token deshalb nur auf sich selbst und die **vorherigen** Tokens schauen, nie in die Zukunft. Das erzwingt eine Maske: Erlaubt sind die Positionen $j \le i$, also das untere Dreieck. Verbotene Scores werden auf $-\infty$ gesetzt, und die Softmax macht daraus exakt 0.

### TODO 3 · Maske bauen
`kausale_maske(n)` gibt eine boolesche $n \times n$-Matrix zurück, die im unteren Dreieck inklusive Diagonale `True` ist. Tipp: `np.tril`.

In [ ]:
def kausale_maske(n):
    # TODO 3
    pass

print(kausale_maske(4))

In [ ]:
_, gew_m = attention(X1, X1, X1, maske=kausale_maske(3)) if kausale_maske(3) is not None else (None, None)
check("TODO 3a Form und Typ", lambda: kausale_maske(4).shape == (4, 4) and kausale_maske(4).dtype == bool)
check("TODO 3b erstes Token sieht nur sich selbst", lambda: np.allclose(gew_m[0], [1, 0, 0]))
check("TODO 3c „Lager“ sieht Spindel und sich selbst", lambda: np.allclose(gew_m[1], [0.5, 0.5, 0]))

## A4 · Query, Key und Value als Projektionen

In einem echten Transformer sind $Q$, $K$ und $V$ nicht einfach $X$. Jedes Token wird mit drei **gelernten** Matrizen projiziert:

$$Q = X W_Q, \qquad K = X W_K, \qquad V = X W_V$$

Anschaulich: Die Query beschreibt, **wonach** ein Token sucht, der Key, **was** es anbietet, und der Value, **welche Information** es weitergibt.

### TODO 4 · Self-Attention mit Projektionen

In [ ]:
def self_attention(X, W_Q, W_K, W_V, maske=None):
    # TODO 4: Q, K, V berechnen und an attention(...) übergeben
    pass

rng = np.random.default_rng(0)
X_bsp = rng.normal(size=(5, 8))                              # 5 Tokens, Modellbreite 8
W_Q, W_K, W_V = (rng.normal(size=(8, 4)) for _ in range(3))  # Projektion auf d_k = 4

In [ ]:
def _referenz(X, wq, wk, wv):
    q, k, v = X @ wq, X @ wk, X @ wv
    s = q @ k.T / 2.0
    w = np.exp(s - s.max(1, keepdims=True)); w /= w.sum(1, keepdims=True)
    return w @ v

check("TODO 4a Ausgabeform (5, 4)", lambda: self_attention(X_bsp, W_Q, W_K, W_V)[0].shape == (5, 4))
check("TODO 4b Ergebnis stimmt", lambda: np.allclose(self_attention(X_bsp, W_Q, W_K, W_V)[0], _referenz(X_bsp, W_Q, W_K, W_V)))

**Multi-Head Attention** (vorgegeben): Mehrere Köpfe mit eigenen Projektionen laufen parallel, jeder kann auf andere Zusammenhänge achten. Die Ergebnisse werden aneinandergehängt und mit $W_O$ zurück auf die Modellbreite projiziert. GPT-2 small hat 12 Köpfe mit je $d_k = 64$, zusammen $12 \cdot 64 = 768$.

In [ ]:
def multi_head(X, koepfe, W_O, maske=None):
    """koepfe: Liste von (W_Q, W_K, W_V) je Kopf."""
    ausgaben = [self_attention(X, wq, wk, wv, maske)[0] for wq, wk, wv in koepfe]
    return np.concatenate(ausgaben, axis=-1) @ W_O

_test = self_attention(X_bsp, W_Q, W_K, W_V)
if _test is not None and _test[0] is not None:
    koepfe = [tuple(rng.normal(size=(8, 4)) for _ in range(3)) for _ in range(2)]   # 2 Köpfe à d_k = 4
    W_O = rng.normal(size=(8, 8))
    print("Eingabe:", X_bsp.shape, "→ Ausgabe:", multi_head(X_bsp, koepfe, W_O, kausale_maske(5)).shape)

## A5 · Warum durch $\sqrt{d_k}$ teilen?

Sind die Komponenten von $q$ und $k$ unabhängig mit Mittelwert 0 und Varianz 1, dann gilt

$$\mathrm{Var}(q \cdot k) = \sum_{i=1}^{d_k} \mathrm{Var}(q_i k_i) = d_k.$$

Bei großem $d_k$ werden die Scores also sehr groß, und die Softmax wird fast zu einem harten Maximum. Die Gradienten verschwinden dann fast. Teilen durch $\sqrt{d_k}$ bringt die Varianz zurück auf 1.

### TODO 5 · Die Statistik nachprüfen
Für jede Dimension $d$ werden 10.000 Paare zufälliger Vektoren erzeugt. Berechnet die Varianz der Skalarprodukte, ohne und mit Skalierung.

In [ ]:
rng = np.random.default_rng(42)
ergebnisse = {}
for d in [4, 64, 512]:
    q = rng.standard_normal((10_000, d))
    k = rng.standard_normal((10_000, d))
    skalarprodukte = np.sum(q * k, axis=1)
    # TODO 5: Varianz ohne Skalierung und nach Teilen durch sqrt(d)
    var_ohne = None
    var_mit = None
    ergebnisse[d] = (var_ohne, var_mit)
    print(f"d = {d:4d}: Varianz ohne Skalierung {var_ohne}, mit Skalierung {var_mit}")

In [ ]:
check("TODO 5a ohne Skalierung ≈ d", lambda: all(abs(v[0] / d - 1) < 0.1 for d, v in ergebnisse.items()))
check("TODO 5b mit Skalierung ≈ 1", lambda: all(abs(v[1] - 1) < 0.1 for v in ergebnisse.values()))

# Was das für die Softmax bedeutet: 8 zufällige Scores bei d = 512
s = np.random.default_rng(1).standard_normal(8) * np.sqrt(512)
print("größtes Gewicht ohne Skalierung:", softmax(s).max().round(3) if softmax(s) is not None else None,
      "| mit Skalierung:", softmax(s / np.sqrt(512)).max().round(3) if softmax(s) is not None else None)

---
# Teil B · GPT-2 von innen

Jetzt ein echtes Modell: GPT-2 small (2019) mit 12 Transformer-Blöcken, 12 Köpfen pro Block und Modellbreite 768. Es ist ein reines **Basismodell**: nur auf Textfortsetzung trainiert, ohne Anweisungs-Training. Es kann vor allem Englisch, deshalb nutzen wir englische Beispiele.

`attn_implementation="eager"` sorgt dafür, dass wir später die Attention-Gewichte auslesen können.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
modell = AutoModelForCausalLM.from_pretrained("openai-community/gpt2", attn_implementation="eager")
modell.eval()

cfg = modell.config
print(f"Blöcke: {cfg.n_layer} | Köpfe je Block: {cfg.n_head} | Modellbreite: {cfg.n_embd} | Kontext: {cfg.n_positions} Tokens")
print(f"Parameter: {sum(p.numel() for p in modell.parameters()) / 1e6:.0f} Mio.")

def naechste_logits(ids):
    """Logits für das nächste Token nach der Token-Folge ids (Liste von ganzen Zahlen)."""
    with torch.no_grad():
        ausgabe = modell(torch.tensor([ids]))
    return ausgabe.logits[0, -1].float().numpy().astype(np.float64)

## B1 · Temperatur und Entropie

Das Modell liefert für jedes der 50.257 Tokens einen Logit. Die **Temperatur** $T$ steuert, wie spitz die Verteilung wird:

$$p_k = \frac{e^{z_k / T}}{\sum_j e^{z_j / T}}$$

Kleines $T$ macht die Verteilung spitzer (fast immer das wahrscheinlichste Token), großes $T$ flacher. Wie „unsicher“ eine Verteilung ist, misst die **Entropie** $H(p) = -\sum_k p_k \log p_k$.

### TODO 6 · Softmax mit Temperatur und Entropie
Für die Entropie gilt $0 \cdot \log 0 = 0$. Lasst Einträge mit $p_k = 0$ deshalb weg (Tipp: `p[p > 0]`).

In [ ]:
def softmax_T(logits, T=1.0):
    # TODO 6a: nutzt eure softmax-Funktion
    pass

def entropie(p):
    # TODO 6b
    pass

In [ ]:
z_bsp = np.array([3.1, 1.9, 1.2, 0.4, -2.0])
check("TODO 6a T = 1 ist die normale Softmax", lambda: np.allclose(softmax_T(z_bsp, 1.0), softmax(z_bsp)))
check("TODO 6a kleines T wird spitzer", lambda: softmax_T(z_bsp, 0.5).max() > softmax_T(z_bsp, 2.0).max())
check("TODO 6b Gleichverteilung über 4 → log 4", lambda: math.isclose(entropie(np.full(4, 0.25)), math.log(4)))
check("TODO 6b sicheres Ergebnis → 0", lambda: math.isclose(entropie(np.array([1.0, 0.0, 0.0])), 0.0, abs_tol=1e-12))

### TODO 7 · Die fünf wahrscheinlichsten nächsten Tokens
Berechnet für den Satzanfang die Wahrscheinlichkeiten (T = 1) und die Indizes `top5` der fünf wahrscheinlichsten Tokens.

In [ ]:
anfang = "The machine stopped because the spindle bearing was"
ids_anfang = tok.encode(anfang)
logits = naechste_logits(ids_anfang)
print("Anzahl Logits:", logits.shape[0])

# TODO 7
p = None
top5 = None

if top5 is not None:
    for i in top5:
        print(f"{p[i]:6.1%}  {tok.decode([int(i)])!r}")

In [ ]:
check("TODO 7a Verteilung über alle Tokens", lambda: p.shape == logits.shape and math.isclose(p.sum(), 1.0))
check("TODO 7b Top 5 absteigend", lambda: len(top5) == 5 and p[top5[0]] == p.max() and p[top5[0]] >= p[top5[4]])

In [ ]:
if softmax_T(logits, 1.0) is not None:
    for T in [0.5, 1.0, 2.0]:
        pT = softmax_T(logits, T)
        best = np.argsort(pT)[::-1][:5]
        print(f"T = {T}: Entropie {entropie(pT):5.2f} | " + ", ".join(f"{tok.decode([int(i)])!r} {pT[i]:.0%}" for i in best))

## B2 · Autoregressive Generierung

Ein Sprachmodell schreibt Text Token für Token: nächstes Token wählen, an die Folge anhängen, wieder vorhersagen. Wählt man immer das wahrscheinlichste Token, heißt das **greedy**. Zieht man zufällig gemäß $p$, heißt das **Sampling**.

### TODO 8 · Die Generierungsschleife
Ergänzt im Schleifenrumpf: Wahrscheinlichkeiten mit Temperatur berechnen, ein Token ziehen (`rng.choice(len(p), p=p)`) und an `ids` anhängen.

In [ ]:
def generiere(text, n_tokens=25, T=0.8, seed=0):
    rng = np.random.default_rng(seed)
    ids = tok.encode(text)
    for _ in range(n_tokens):
        logits = naechste_logits(ids)
        # TODO 8: p berechnen, Token ziehen, an ids anhängen
        pass
    return tok.decode(ids)

print(generiere("The maintenance team found that the spindle bearing"))

In [ ]:
start = "The maintenance team found that the spindle bearing"
check("TODO 8a Text wird länger", lambda: len(tok.encode(generiere(start, n_tokens=5))) >= len(tok.encode(start)) + 4)
check("TODO 8b gleicher Seed → gleicher Text", lambda: generiere(start, 5, seed=3) == generiere(start, 5, seed=3) != start)

if len(tok.encode(generiere(start, n_tokens=2))) > len(tok.encode(start)):
    for T in [0.3, 1.0, 1.8]:
        print(f"\nT = {T}: {generiere(start, n_tokens=25, T=T, seed=1)}")

> **Zum Nachdenken:** Bei kleinem $T$ wiederholt sich der Text oft, bei großem wird er wirr. Welche Temperatur würdet ihr für ein Instandhaltungsprotokoll wählen, welche für Ideen für eine Produktbezeichnung?

## B3 · In GPT-2 hineinschauen: Attention-Gewichte

Führt die Zelle aus und ändert `schicht` (0 bis 11) und `kopf` (0 bis 11). Achtet auf das leere obere Dreieck: die kausale Maske aus TODO 3.

In [ ]:
satz = "The bearing of the spindle is hot because the lubrication failed"
ids_satz = tok.encode(satz)
woerter = [tok.decode([i]) for i in ids_satz]
with torch.no_grad():
    aus = modell(torch.tensor([ids_satz]), output_attentions=True)

schicht, kopf = 5, 1
A = aus.attentions[schicht][0, kopf].float().numpy()
plt.figure(figsize=(7, 6))
plt.imshow(A, cmap="viridis")
plt.xticks(range(len(woerter)), woerter, rotation=90); plt.yticks(range(len(woerter)), woerter)
plt.xlabel("schaut auf (Key)"); plt.ylabel("Token (Query)"); plt.colorbar()
plt.title(f"GPT-2 · Schicht {schicht} · Kopf {kopf}")
plt.show()
print("Zeilensummen (sollten 1 sein):", np.round(A.sum(axis=1), 3))

> **Vorsicht beim Deuten:** Viele Köpfe schauen auffällig stark auf das erste Token oder auf das direkte Vorgängertoken. Einzelne Köpfe sind nur ein kleiner Ausschnitt davon, was das Modell tut.

## B4 · Ausblick: Warum befolgt GPT-2 keine Anweisungen?

In [ ]:
print(generiere("Translate into German: The spindle bearing is hot.\nGerman:", n_tokens=20, T=0.7, seed=2))

GPT-2 setzt Text fort, wie er im Internet stehen könnte. Ein Assistent wie Mistral Small 4 aus der KI-Toolbox befolgt die Anweisung. Den Unterschied macht das Training **nach** dem Pretraining. Darum geht es in Woche 6.

Optional mit Toolbox-Schlüssel: dieselbe Anweisung an ein Modell der KI-Toolbox.

In [ ]:
MODELL = "kit.mistral-small-4-119b-a8b"
try:
    from openai import OpenAI
    try:
        from google.colab import userdata
        API_KEY = userdata.get("KIT_TOOLBOX_KEY")
    except ImportError:
        import os
        API_KEY = os.environ.get("KIT_TOOLBOX_KEY")
    client = OpenAI(api_key=(API_KEY or "fehlt").strip(), base_url="https://ki-toolbox.scc.kit.edu/api/v1")
    antwort = client.chat.completions.create(model=MODELL, max_tokens=50,
                                             messages=[{"role": "user", "content": "Translate into German: The spindle bearing is hot."}])
    print("Mistral Small 4:", antwort.choices[0].message.content)
except Exception as e:
    client = None
    print("Übersprungen:", type(e).__name__)

**Temperatur am großen Modell.** Die Modelle der KI-Toolbox nehmen den Parameter `temperature` direkt an. Die Zelle stellt dieselbe offene Frage je dreimal bei $T = 0$ und bei $T = 1{,}2$. Wie viele verschiedene Antworten bekommt ihr jeweils?

In [ ]:
FRAGE = "Erfinde einen kurzen Namen für einen Roboter in der Instandhaltung. Antworte nur mit dem Namen."
if client is not None:
    for T in [0.0, 1.2]:
        namen = [client.chat.completions.create(model=MODELL, max_tokens=20, temperature=T,
                                                messages=[{"role": "user", "content": FRAGE}]).choices[0].message.content.strip()
                 for _ in range(3)]
        print(f"T = {T}: {namen} → {len(set(namen))} verschiedene")
else:
    print("Übersprungen: kein Zugang zur KI-Toolbox")

> **Zum Vergleich:** Nicht jeder Anbieter erlaubt das. Bei kommerziellen Claude-Modellen ab Version 4.7 lassen sich `temperature`, `top_p` und `top_k` laut Anthropic nicht mehr frei wählen, dort steuert man das Verhalten über den Prompt (Woche 6). Bei offenen Modellen wie GPT-2 und den Modellen der KI-Toolbox könnt ihr die Stellschrauben dagegen selbst ausprobieren.

---
## Bonus

**B1 · Top-p-Sampling.** Schreibt `top_p_filter(p, schwelle=0.9)`: Sortiert die Wahrscheinlichkeiten absteigend, behaltet die kleinste Menge, deren Summe die Schwelle erreicht, setzt den Rest auf 0 und normiert neu. Nutzt sie in `generiere`.

**B2 · Köpfe suchen.** Findet einen Kopf in GPT-2, der fast immer auf das direkte Vorgängertoken schaut. Tipp: Mittelwert der Nebendiagonale `np.diag(A, k=-1)` für alle Schichten und Köpfe vergleichen.

In [ ]:
# B1 / B2

## Zusammenfassung

- **Attention** mischt die Vektoren aller Tokens, gewichtet nach Ähnlichkeit von Query und Key: $\mathrm{softmax}(QK^\top/\sqrt{d_k})V$. So bekommt dasselbe Wort je nach Kontext einen anderen Vektor.
- $Q$, $K$ und $V$ entstehen durch gelernte Projektionen. **Multi-Head** lässt mehrere Köpfe parallel auf verschiedene Zusammenhänge achten.
- Die **kausale Maske** verhindert den Blick in die Zukunft. Das **Teilen durch** $\sqrt{d_k}$ hält die Varianz der Scores bei 1.
- Ein Sprachmodell erzeugt Text **autoregressiv**. Die **Temperatur** steuert, wie zufällig die Auswahl ist, die **Entropie** misst die Unsicherheit.

### Ausblick auf Woche 6
GPT-2 setzt Text fort, ein Assistent befolgt Anweisungen. Was passiert zwischen Pretraining und Assistent? Und wie schreibt man Prompts, die zuverlässig das gewünschte Format liefern?